# Arm B: LoRA fine-tune of Qwen2.5-Coder-0.5B on Jahanvi's corpus

Two phases on a T4:
1. **LM adaptation** — causal-LM LoRA pass over the full personal code corpus (the 'trained on everything I've written' phase).
2. **SFT** — chat pairs mapping NL descriptions to Web VPython sims (her 4 training sims oversampled + templated pairs for the public demos).

Then: merge adapter, convert to GGUF, quantize Q4_K_M, download for llama.cpp/Ollama.

Before running: zip corpus + data on the laptop:
`Compress-Archive -Path corpus/raw, data -DestinationPath arm_b_data.zip`


In [ ]:
!nvidia-smi -L
!pip -q install transformers peft datasets accelerate trl


In [ ]:
!git clone https://github.com/JahanviChamria/local-llm-webvpython.git
%cd local-llm-webvpython
from google.colab import files
up = files.upload()  # arm_b_data.zip
!unzip -oq arm_b_data.zip


## Phase 1: LM adaptation on the full corpus


In [ ]:
import torch, glob
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model

BASE = 'Qwen/Qwen2.5-Coder-0.5B-Instruct'
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.bfloat16, device_map='auto')

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()


In [ ]:
texts = []
for f in glob.glob('corpus/raw/general/*') + glob.glob('corpus/raw/sims/*') + glob.glob('corpus/raw/vpython-public/*'):
    if f.endswith('manifest.json'): continue
    texts.append(open(f, encoding='utf-8', errors='ignore').read())

# skip the held-out eval sims
import json
heldout = set(json.load(open('data/split.json'))['heldout'])
texts = [t for f, t in zip(
    glob.glob('corpus/raw/general/*') + glob.glob('corpus/raw/sims/*') + glob.glob('corpus/raw/vpython-public/*'), texts)
    if f.split('/')[-1] not in heldout and not f.endswith('manifest.json')]

BLOCK = 1024
ids = sum([tok(t)['input_ids'] + [tok.eos_token_id] for t in texts], [])
blocks = [ids[i:i+BLOCK] for i in range(0, len(ids)-BLOCK, BLOCK)]
ds = Dataset.from_dict({'input_ids': blocks})
print(len(blocks), 'blocks')


In [ ]:
args = TrainingArguments(output_dir='lora_phase1', per_device_train_batch_size=4,
    gradient_accumulation_steps=4, num_train_epochs=1, learning_rate=2e-4,
    lr_scheduler_type='cosine', warmup_ratio=0.03, logging_steps=10, save_strategy='no',
    bf16=True, report_to='none')
Trainer(model=model, args=args, train_dataset=ds,
        data_collator=DataCollatorForLanguageModeling(tok, mlm=False)).train()


## Phase 2: SFT on description -> sim pairs


In [ ]:
import json
pairs = [json.loads(l) for l in open('data/lora_train.jsonl', encoding='utf-8')]

# templated pairs for python-syntax public demos, weak labels but useful signal
demo_pairs = []
for f in glob.glob('corpus/raw/vpython-public/*.py'):
    name = f.split('__')[-1].replace('.py','').replace('-VPython','')
    if 'JavaScript' in f: continue
    src = open(f, encoding='utf-8').read()
    src = '
'.join(l for l in src.splitlines() if not l.startswith(('GlowScript','Web VPython'))).strip()
    demo_pairs.append({'messages': [
        {'role':'system','content': pairs[0]['messages'][0]['content']},
        {'role':'user','content': f'Write the classic GlowScript example program {name} in Web VPython.'},
        {'role':'assistant','content': src}]})

sft_data = pairs * 10 + demo_pairs   # oversample her sims
sft_texts = [tok.apply_chat_template(p['messages'], tokenize=False) for p in sft_data]
sft_ds = Dataset.from_dict({'text': sft_texts})
print(len(sft_ds), 'SFT examples')


In [ ]:
from trl import SFTTrainer, SFTConfig
sft_args = SFTConfig(output_dir='lora_phase2', per_device_train_batch_size=2,
    gradient_accumulation_steps=8, num_train_epochs=3, learning_rate=1e-4,
    lr_scheduler_type='cosine', warmup_ratio=0.03, logging_steps=5, save_strategy='no',
    bf16=True, report_to='none', max_length=2048, dataset_text_field='text')
SFTTrainer(model=model, args=sft_args, train_dataset=sft_ds).train()


## Smoke test


In [ ]:
prompt = tok.apply_chat_template([
    {'role':'system','content': pairs[0]['messages'][0]['content']},
    {'role':'user','content': 'A red ball bounces on a blue floor under gravity, leaving a trail.'}],
    tokenize=False, add_generation_prompt=True)
out = model.generate(**tok(prompt, return_tensors='pt').to(model.device), max_new_tokens=300, do_sample=True, temperature=0.7)
print(tok.decode(out[0][len(tok(prompt)['input_ids']):], skip_special_tokens=True))


## Merge, convert to GGUF, quantize


In [ ]:
merged = model.merge_and_unload()
merged.save_pretrained('qwen-vpython-merged')
tok.save_pretrained('qwen-vpython-merged')


In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip -q install -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python llama.cpp/convert_hf_to_gguf.py qwen-vpython-merged --outfile qwen-vpython-f16.gguf --outtype f16


In [ ]:
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF > /dev/null
!cmake --build llama.cpp/build --target llama-quantize -j 2 > /dev/null
!llama.cpp/build/bin/llama-quantize qwen-vpython-f16.gguf qwen-vpython-q4_k_m.gguf Q4_K_M
!ls -lh *.gguf


In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/local-llm-webvpython
!cp qwen-vpython-q4_k_m.gguf /content/drive/MyDrive/local-llm-webvpython/
files.download('qwen-vpython-q4_k_m.gguf')
